In [7]:
%matplotlib widget
# Add the directory containing the package to sys.path
import sys, os
package_dir = os.path.abspath("C:/Users/froll/Documents/Labo/Projets/Outils/swd")
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)
    
from swd import spherical_processing as sp
import importlib

import numpy as np
from numpy import pi
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm
np.set_printoptions(precision=2, suppress=True)

# Enable LaTeX rendering
plt.rc('text', usetex=True)
# Improve resolution
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

# Import utils_SH_selection
scripts_dir = os.path.abspath("C:/Users/froll/Documents/Labo/Projets/Violon/scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

import utils_SH_selection
importlib.reload(utils_SH_selection)

<module 'utils_SH_selection' from 'C:\\Users\\froll\\Documents\\Labo\\Projets\\Violon\\scripts\\utils_SH_selection.py'>

In [8]:
#Vitesse du son au moment de la mesure, dependant de la temperature:
Tc = 21.5 
C = np.sqrt( 1.4 * 287 *(Tc + 273) )
Path = './'
NbMems = 256
NbViolTot = 6
NbViol = 6

NumViolon = np.load('./../results/NumViolon.npz')['NumViolon']
XYZViolins = np.load('./../results/XYZViolinsAligned.npy')

NbMics = 256
NbSrcs = XYZViolins.shape[0] - NbMics

XYZHammerImpact = XYZViolins[NbSrcs-1]
XYZ = XYZViolins-XYZHammerImpact
XYZs = XYZ[:NbSrcs-1]
XYZm = XYZ[NbSrcs:]
XYZHammerImpact = XYZ[NbSrcs-1]

Rm = np.linalg.norm(XYZm,axis=1)
Rmin = np.min(Rm)

### Array Quality Estimation

In [9]:
from ArrayPerfs import *
print(process_array_quality(XYZm, 1000))

Max Order (N): 15
Condition Number: 4.28e+11
WNG (dB): -110.77
Directivity Index (dB): 24.08
Robustness: Low


In [10]:
Pm = np.load('./../results/ViolinsFRFsAndRIs.npz')['Hhm']
frq = np.load('./../results/ViolinsFRFsAndRIs.npz')['frq']

MAX_ORDER = 15
NbSHMax = (MAX_ORDER+1)**2

frqmin = 100
frqMax = 12500

iBand = np.where((frq>=frqmin) & (frq<=frqMax))[0]
Band = frq[iBand]

Dyn = 36
CyclicScale = 'icefire' #edge, icefire, phase, hsv
RealScale = 'seismic'
Magnitudescale = 'inferno'

## Process optimal $\lambda$ and SH truncation order at all frequencies for each Violin

In [11]:
H_max = sp.compute_SphericalWavesbasis_origin_to_field(XYZm, Band, MAX_ORDER, SH_center=np.array([0,0,0]))

### Mathematical Description of the Optimal $C_{mn}$ Process

The acoustic pressure $P(\mathbf{r}, \omega)$ measured at $M$ microphone positions can be expanded into a set of Spherical Harmonics (SH) up to a truncation order $N$:
$$ \mathbf{P} \approx \mathbf{H}_N \mathbf{C}_N $$
where $\mathbf{P} \in \mathbb{C}^{M \times 1}$ is the measured pressure vector, $\mathbf{H}_N \in \mathbb{C}^{M \times (N+1)^2}$ is the transfer matrix (Spherical Wave basis), and $\mathbf{C}_N \in \mathbb{C}^{(N+1)^2 \times 1}$ contains the unknown SH coefficients.

#### 1. Regularized Inverse Problem
Due to the ill-posed nature of the problem (especially at low frequencies or high truncation orders), we use Tikhonov regularization to estimate the coefficients against measurement noise and ill-conditioning:
$$ \mathbf{C}_N(\lambda) = \arg\min_{\mathbf{C}} \left( ||\mathbf{P}_{meas} - \mathbf{H}_N \mathbf{C}||_2^2 + \lambda ||\mathbf{C}||_2^2 \right) $$
where $\lambda$ is the regularization parameter.

#### 2. K-Fold Cross-Validation
To select the optimal pair $(N_{opt}, \lambda_{opt})$ that generalizes well and avoids spatial overfitting, a $K$-fold Cross-Validation (CV) is performed:
1. The $M$ microphones are randomly partitioned into $K$ disjoint validation sets $\mathcal{V}_k$.
2. For each fold $k$, the remaining microphones form the interpolation set $\mathcal{I}_k$.
3. The coefficients $\mathbf{C}_N^{(k)}(\lambda)$ are estimated using only the interpolation set:
   $$ \mathbf{C}_N^{(k)}(\lambda) = (\mathbf{H}_{\mathcal{I}_k}^H \mathbf{H}_{\mathcal{I}_k} + \lambda \mathbf{I})^{-1} \mathbf{H}_{\mathcal{I}_k}^H \mathbf{P}_{\mathcal{I}_k} $$
4. The pressure is then predicted at the validation points:
   $$ \mathbf{P}_{rec}^{(k)} = \mathbf{H}_{\mathcal{V}_k} \mathbf{C}_N^{(k)}(\lambda) $$
5. The overall Relative Reconstruction Error $\mathcal{L}(N, \lambda)$ is computed across all folds by comparing predicted and measured validation pressures.

#### 3. Optimal Parameter Selection
We select parameters that produce minimal reconstruction error while prioritizing solutions with better matrix conditioning. The condition number of the transfer matrix $\kappa(\mathbf{H}_N)$ dictates the noise amplification.
The optimal parameters are selected by:
1. Finding the minimum CV error: $\mathcal{L}_{min} = \min_{N, \lambda} \mathcal{L}(N, \lambda)$
2. Defining a subset of acceptable parameters that yield an error close to the minimum (within a tolerance threshold $\epsilon$, e.g., $\epsilon = 0.05$):
   $$ \mathcal{S}_{accept} = \{ (N, \lambda) \mid \mathcal{L}(N, \lambda) \le \mathcal{L}_{min} + \epsilon \} $$
3. Among this subset of valid models, selecting the one that minimizes the condition number $\kappa(\mathbf{H}_N)$ to guarantee maximum robustness:
   $$ (N_{opt}, \lambda_{opt}) = \arg\min_{(N, \lambda) \in \mathcal{S}_{accept}} \kappa(\mathbf{H}_N) $$

This cross-validation procedure provides the most stable, parsimonious Spherical Harmonic expansion capable of accurately reconstructing the sound field.

In [ ]:
Process_Opt_Lbda = True # Set to False to load previously computed parameters instead of re-optimizing

target_freqs = Band # Use the actual frequencies selected
results_freqs = list(target_freqs)
Nfreq = len(target_freqs)
kvect_study = 2 * np.pi * np.array(target_freqs) / C

truncation_order_list = np.arange(MAX_ORDER + 1)
N_truncation_order = len(truncation_order_list)
lambda_reg_list = [0]
Nlambda = len(lambda_reg_list)

if Process_Opt_Lbda:
    # Initialize storage
    results_lambdas = {v: np.zeros(Nfreq) for v in range(NbViol)}
    results_orders = {v: np.zeros(Nfreq, dtype=int) for v in range(NbViol)}
    L_opt = {v: np.zeros(Nfreq) for v in range(NbViol)}

    print(f"Optimizing orders for {NbViol} violins over {Nfreq} frequencies...")

    N_validation_sets = 5
    Narray = NbMics
    Narray_validation = Narray // N_validation_sets
    validation_sets = [np.random.choice(Narray, size=Narray_validation, replace=False) for _ in range(N_validation_sets)]

    kappa = np.empty((N_truncation_order, Nfreq), dtype=float)
    for ind_truncation_order, NSH_analysis in enumerate(truncation_order_list):
        for ind_freq in range(Nfreq):
            kappa[ind_truncation_order, ind_freq] = np.linalg.cond(H_max[:, :(NSH_analysis + 1)**2, ind_freq])

    for v in tqdm(range(NbViol), desc="Violins"):
        x_meas = Pm[v, iBand, :].T # shape: (Narray, Nfreq)
        
        L_versus_N = np.empty((N_truncation_order, Nfreq), dtype=float)

        for ind_truncation_order, NSH_analysis in enumerate(tqdm(truncation_order_list, desc="Orders", leave=False)):
            for ind_lambda, lambda_reg in enumerate(lambda_reg_list):
                x_rec = np.empty((Narray, Nfreq), dtype=complex)
                for ind_set in range(N_validation_sets):
                    validation_indices = validation_sets[ind_set]
                    x_meas_validation = x_meas[validation_indices, :]
                    interpolation_indices = [ind for ind in np.arange(Narray) if ind not in validation_indices]
                    x_meas_interpolation = x_meas[interpolation_indices, :]
                    
                    H_array = H_max[interpolation_indices, :(NSH_analysis + 1)**2, :]
                    H_origin_to_field = H_max[validation_indices, :(NSH_analysis + 1)**2, :]
                    
                    for ind_freq in range(Nfreq):
                        H_array_f = H_array[:, :, ind_freq]
                        H_origin_to_field_f = H_origin_to_field[:, :, ind_freq]
                        x_meas_interpolation_f = x_meas_interpolation[:, ind_freq][:, None]
                        
                        cmn = sp.compute_SHcoefs(x_meas_interpolation_f, H_array_f, lambda_reg=lambda_reg)
                        x_rec[validation_indices, ind_freq] = (H_origin_to_field_f @ cmn).squeeze()
                
                L_versus_N[ind_truncation_order, :] = np.linalg.norm(x_meas - x_rec, axis=0) / np.linalg.norm(x_meas, axis=0)

        threshold_min = 5e-2

        for ind_freq in range(Nfreq):
            L_versus_N_list = L_versus_N[:, ind_freq]
            kappa_list = kappa[:, ind_freq]
            
            Lmin = np.min(L_versus_N_list)
            ind_close_to_min = np.where(L_versus_N_list < Lmin + threshold_min)[0]
            ind_opt_kappa = np.min(ind_close_to_min) + np.argmin(kappa_list[ind_close_to_min])
            
            results_orders[v][ind_freq] = truncation_order_list[ind_opt_kappa]
            results_lambdas[v][ind_freq] = lambda_reg_list[0] # Extensible to multiple lambdas
            L_opt[v][ind_freq] = L_versus_N_list[ind_opt_kappa]

    np.savez(f'./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz', 
            orders=results_orders, 
            lambdas=results_lambdas, 
            freqs=results_freqs)
    print(f"Saved optimization parameters to ./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz")
else:
    data = np.load(f'./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz', allow_pickle=True)
    results_freqs = data['freqs']
    results_lambdas = data['lambdas'].item()
    results_orders = data['orders'].item()
    print(f"Loaded parameters from ./../results/Optimal_Lambdas_Orders_O{MAX_ORDER}.npz")

fig, ax = plt.subplots(1, 1, figsize=(6, 3))

viol_names = NumViolon if 'NumViolon' in locals() else [f"Violin {v}" for v in range(NbViol)]
cm_plot = plt.get_cmap('gist_rainbow') 
colors = [cm_plot(1.*i/NbViol) for i in range(NbViol)]

for v in range(NbViol):
    lbl = viol_names[v] if v < len(viol_names) else f"Violin {v}"
    ax.semilogx(results_freqs, results_orders[v], '-', linewidth=1.5, label=lbl, color=colors[v])

ax.set_xlim(frqmin, frqMax)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Optimal SH Order')
ax.set_title('Optimal SH Truncation Order')
ax.grid(True, which='both', alpha=0.8)
ax.set_yticks(np.arange(0, MAX_ORDER + 2, 1))
ax.legend(ncol=2, fontsize='small')

plt.tight_layout()
plt.show()


Optimizing orders for 6 violins over 2481 frequencies...


In [ ]:

print(f"\nComputing Cmn for all {NbViol} violins using compute_SHcoefs_VariableLbdas...")

NbSH_max = (MAX_ORDER + 1)**2
Cmn_all = np.zeros((NbViol, NbSH_max, len(results_freqs)), dtype=complex)

kvect = 2 * np.pi * np.array(results_freqs) / C
# Compute basis up to MAX_ORDER for all frequencies at once
H_max = sp.compute_SphericalWavesbasis_origin_to_field(
    XYZm, kvect, MAX_ORDER, SH_center=np.array([0,0,0])
)

for v in tqdm(range(NbViol), desc="Processing violins"):
    order_v = np.array(results_orders[v], dtype=int)
    lam_v = np.array(results_lambdas[v])
    
    # pmeas should be (N_fieldpoints, N_k)
    pmeas_v = Pm[v, iBand, :].T 
    
    # Use swd2 dedicated function
    c_coeffs = sp.compute_SHcoefs_VariableLbdas(
        pmeas=pmeas_v, 
        H=H_max, 
        N_SH_vect=order_v, 
        lambda_reg=lam_v
    )
    
    Cmn_all[v, :, :] = c_coeffs

# Update Globals for consistency (Set Cmn0 to the selected violin NumV)
NumV = 1
Cmn0 = Cmn_all[NumV]
OSH = MAX_ORDER
O_SH_vect = np.array(results_orders[NumV])
print(f"Cmn computation complete. Shape: {Cmn_all.shape}")

In [ ]:
np.savez('./../results/CmnO'+str(MAX_ORDER)+'_Optimization_Results.npz', 
         orders=results_orders, 
         lambdas=results_lambdas, 
         freqs=results_freqs,
         Cmn_all=Cmn_all)
print("Saved detailed optimization results to ./../results/CmnO"+str(MAX_ORDER)+"_Optimization_Results.npz")